# 01 - Data Collection

This notebook covers the full scraping pipeline from raw URLs to clean CSVs. It consolidates the initial demo scraping, the Fox-specific cleaning investigation, and the NBC-specific cleaning investigation.

**Datasets produced:**
- `data/raw/fox_scraped_all.csv` — 2,000 Fox News articles
- `data/raw/nbc_scraped_all.csv` — 1,805 NBC News articles

**Sections:**
1. URL inventory — understanding what we're working with
2. Scraping approach and implementation
3. Bugs discovered and fixed (Fox: `.print` URLs, `/live-news` pages)
4. Fox News dataset: NA investigation
5. NBC News dataset: NA investigation
6. Final dataset summary

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import random

## 1. URL Inventory

The course provided 3,815 article URLs in `url_only_data.csv`. We start by understanding the split and structure.

In [2]:
links = pd.read_csv('../data/external/url_only_data.csv')

fox_urls = links[links['url'].str.contains('foxnews.com', na=False)]
nbc_urls = links[links['url'].str.contains('nbcnews.com', na=False)]

print(f'Total URLs: {len(links):,}')
print(f'Fox News:   {len(fox_urls):,}')
print(f'NBC News:   {len(nbc_urls):,}')

# Sample a few to understand URL structure
print('\nFox URL samples:')
for u in fox_urls['url'].head(5).tolist():
    print(' ', u)

print('\nNBC URL samples:')
for u in nbc_urls['url'].head(5).tolist():
    print(' ', u)

Total URLs: 3,805
Fox News:   2,000
NBC News:   1,805

Fox URL samples:
  https://www.foxnews.com/lifestyle/jack-carrs-eisenhower-d-days-memo-noble-undertaking
  https://www.foxnews.com/entertainment/bruce-willis-demi-moore-avoided-doing-one-thing-while-co-parenting-daughter-says
  https://www.foxnews.com/politics/blinken-meets-with-qatars-prime-minister.print
  https://www.foxnews.com/entertainment/emily-blunt-says-toes-curl-when-people-their-kids-want-act-want-say-dont-do-it
  https://www.foxnews.com/media/the-view-co-host-cnn-commentator-ana-navarro-host-night-2-democratic-national-convention

NBC URL samples:
  https://www.nbcnews.com/news/world/helicopter-carrying-irans-president-suffers-hard-landing-state-tv-says-rcna152961
  https://www.nbcnews.com/pop-culture/celebrity/kristen-cavallari-husband-jay-cutler-divorce-after-10-years-together-n1192966
  https://www.nbcnews.com/news/asian-america/why-atlanta-spa-shooter-s-asian-acquaintances-can-t-tell-n1275348
  https://www.nbcnews.c

## 2. Scraping Approach

We use `requests` + `BeautifulSoup`. The target fields are:
- **title** — the `<h1>` headline
- **subtitle** — the article dek/sub-headline
- **author** — byline
- **datetime_posted** — ISO timestamp from `<time>` tag
- **topic/label** — source identifier

We include random delays of 0.5–1.5s between requests and exponential backoff on rate-limit responses. The production version of this code lives in `scripts/scraper.py`.

In [3]:
# Quick single-URL demo (from the course example)
url = 'https://www.foxnews.com/sports/juan-soto-sends-yankees-worldseries-first-time-15-years'

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}
response = requests.get(url, headers=headers, timeout=10)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    title = soup.find('h1', class_='headline speakable')
    print('Title:', title.get_text(strip=True) if title else 'Not found')
else:
    print(f'HTTP {response.status_code}')

HTTP 404


## 3. Bugs Discovered and Fixed

During initial scraping, two categories of Fox URLs caused significant NA rates. These were diagnosed and fixed before the final scrape run.

### Bug 1: `.print` suffix URLs (443 Fox URLs)

Fox News appends `.print` to some article URLs (e.g., `foxnews.com/politics/article.print`). These return a stripped printer-friendly page with no headline tags — the standard `<h1 class="headline speakable">` selector returns nothing.

**Fix:** Strip `.print` from all URLs before fetching.

### Bug 2: `/live-news` pages (10 Fox URLs)

Live coverage pages at `foxnews.com/live-news/...` use a completely different HTML structure — there is no `headline speakable` class. The page is rendered more dynamically.

**Fix:** Detect `/live-news` in the URL and route to a dedicated scraping function that looks for a different headline selector.

In [4]:
# Demonstrate the .print issue
print_urls = [u for u in fox_urls['url'].tolist() if u.endswith('.print')]
print(f'Fox URLs ending in .print: {len(print_urls)}')
print('\nSamples:')
for u in print_urls[:5]:
    print(' ', u)

live_urls = [u for u in fox_urls['url'].tolist() if '/live-news' in u]
print(f'\nFox /live-news URLs: {len(live_urls)}')

Fox URLs ending in .print: 443

Samples:
  https://www.foxnews.com/politics/blinken-meets-with-qatars-prime-minister.print
  https://www.foxnews.com/politics/striking-boeing-workers-boo-after-democrat-sen-maria-cantwell-criticizes-trump.print
  https://www.foxnews.com/politics/white-house-grilled-harris-gun-ownership-mandatory-gun-buybacks.print
  https://www.foxnews.com/food-drink/pennsylvania-family-sickened-after-eating-toxic-mushrooms-expected-recover.print
  https://www.foxnews.com/health/family-selling-dream-home-fund-life-saving-treatment-daughter.print

Fox /live-news URLs: 10


## 4. Fox News Dataset: NA Investigation

After running the fixed scraper, we load the output and investigate remaining NAs.

In [5]:
fox = pd.read_csv('../data/raw/fox_scraped_all.csv')

print('Shape:', fox.shape)
print('\nNA counts:')
print(fox.isna().sum())

Shape: (2000, 7)

NA counts:
url                 0
topic               0
title               0
subtitle           13
author              0
datetime_posted     0
label               0
dtype: int64


In [6]:
# Investigate the 13 subtitle NAs
subtitle_na_urls = fox[fox['subtitle'].isna()]['url'].values
print(f'Articles with missing subtitle: {len(subtitle_na_urls)}')
print('\nConclusion: manual inspection confirms these 13 articles genuinely have no subtitle.')
print('These rows are kept as-is — the scraper is working correctly.')

# Show which topics they fall into
print('\nCategories of subtitle-NA rows:')
fox_na = fox[fox['subtitle'].isna()].copy()
fox_na['category'] = fox_na['url'].str.extract(r'foxnews\.com/([^/]+)')
print(fox_na['category'].value_counts())

Articles with missing subtitle: 13

Conclusion: manual inspection confirms these 13 articles genuinely have no subtitle.
These rows are kept as-is — the scraper is working correctly.

Categories of subtitle-NA rows:
category
entertainment    6
politics         2
science          1
opinion          1
media            1
world            1
us               1
Name: count, dtype: int64


In [7]:
# Topic column investigation — the scraped 'topic' values are article-level pills,
# not a consistent taxonomy. 314 unique values is not useful for modeling.
print(f"Unique 'topic' values: {fox['topic'].nunique()}")
print('Top 10:')
print(fox['topic'].value_counts().head(10))

# Better: extract from URL
fox['category'] = fox['url'].str.extract(r'foxnews\.com/([^/]+)')
print('\nURL-extracted category distribution:')
print(fox['category'].value_counts())

Unique 'topic' values: 314
Top 10:
topic
Media            231
POLITICS         196
LIFESTYLE        148
Donald Trump      76
Israel            58
HEALTH            55
ENTERTAINMENT     53
Kamala Harris     53
Joe Biden         38
OPINION           38
Name: count, dtype: int64

URL-extracted category distribution:
category
politics          615
media             385
lifestyle         203
us                201
entertainment     166
world             110
sports             90
health             73
travel             41
opinion            39
food-drink         35
official-polls     14
tech               12
live-news          10
faith-values        3
science             1
auto                1
great-outdoors      1
Name: count, dtype: int64


In [8]:
# Verify datetime format
print('Sample datetime_posted values:')
print(fox['datetime_posted'].head(5).tolist())

# Check top authors
print('\nTop 5 authors:')
print(fox['author'].value_counts().head(5))

Sample datetime_posted values:
['2023-10-13T14:06:08-04:00', '2024-10-18T15:56:05-04:00', '2024-08-19T21:00:35-04:00', '2023-06-09T13:55:28-04:00', '2024-06-06T04:30:24-04:00']

Top 5 authors:
author
Joseph A. Wulfsohn    69
Paul Steinhauser      66
Kerry J. Byrne        63
Melissa Rudy          51
Julia Johnson         47
Name: count, dtype: int64


## 5. NBC News Dataset: NA Investigation

In [9]:
nbc = pd.read_csv('../data/raw/nbc_scraped_all.csv')

# Verify we scraped all the NBC URLs
nbc_url_count = len(links[links['url'].str.contains('nbcnews.com', na=False)])
print(f'Expected NBC rows: {nbc_url_count}')
print(f'Actual NBC rows:   {len(nbc)}')
print(f'Match: {nbc_url_count == len(nbc)}')

print('\nNA counts:')
print(nbc.isna().sum())

Expected NBC rows: 1805
Actual NBC rows:   1805
Match: True

NA counts:
url                 0
topic               0
title               4
subtitle            4
author             96
datetime_posted    94
label               0
dtype: int64


In [10]:
# 4 NAs in title/subtitle — investigate
broken_urls = nbc[nbc['title'].isna()]['url'].values
print(f'Articles with missing titles: {len(broken_urls)}')
print('\nURLs:')
for u in broken_urls:
    print(' ', u)
print('\nConclusion: all 4 are /select/ shopping pages that no longer load.')
print('These will be dropped before modeling (4/1805 = 0.2% loss).')

Articles with missing titles: 4

URLs:
  https://www.nbcnews.com/select/shopping/select-pet-awards-under-25-ncna1306012
  https://www.nbcnews.com/feature/nbc-out/lil-nas-x-dolly-parton-lena-waithe-appear-virtual-glaad-n1233424
  https://www.nbcnews.com/select/shopping/new-notable-sonos-jbl-fitbit-rcna156060
  https://www.nbcnews.com/select/shopping/mr-coffee-mug-warmer-gift-ncna1285511

Conclusion: all 4 are /select/ shopping pages that no longer load.
These will be dropped before modeling (4/1805 = 0.2% loss).


In [11]:
# 96 author NAs + 94 datetime NAs — investigate
# Are these the same rows?
author_na = set(nbc[nbc['author'].isna()].index)
dt_na = set(nbc[nbc['datetime_posted'].isna()].index)
print(f'Author NAs: {len(author_na)}')
print(f'Datetime NAs: {len(dt_na)}')
print(f'Overlap: {len(author_na & dt_na)}')

# Check what kind of pages these are
nbc_author_na_urls = nbc[nbc['author'].isna() & nbc['title'].notna()]['url'].head(5).tolist()
print('\nSample author-NA URLs:')
for u in nbc_author_na_urls:
    print(' ', u)
print('\nConclusion: these are /live-blog/ pages — live news update threads.')
print('They have titles but no single author byline or datetime.')
print('Kept in dataset — titles are usable for classification. Missing datetimes go to train-only.')

Author NAs: 96
Datetime NAs: 94
Overlap: 94

Sample author-NA URLs:
  https://www.nbcnews.com/politics/supreme-court/live-blog/trump-immunity-supreme-court-ruling-live-updates-rcna159539
  https://www.nbcnews.com/politics/2024-election/live-blog/dnc-2024-live-updates-rcna165226/rcrd52541?canonicalCard=true
  https://www.nbcnews.com/politics/donald-trump/live-blog/live-updates-trump-indictment-classified-documents-rcna88494
  https://www.nbcnews.com/politics/live-blog/jan-6-hearing-expect-top-trump-admin-aide-testifies-rcna35551
  https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-rcna141090/rcrd35599?canonicalCard=true

Conclusion: these are /live-blog/ pages — live news update threads.
They have titles but no single author byline or datetime.
Kept in dataset — titles are usable for classification. Missing datetimes go to train-only.


## 6. Final Dataset Summary

In [12]:
print('══ FINAL SCRAPE SUMMARY ══════════════════════════════')
print(f'Fox News articles:           {len(fox):,}')
print(f'  - Missing subtitle:        {fox["subtitle"].isna().sum()} (articles with no dek)')
print(f'  - Missing title:           {fox["title"].isna().sum()}')
print()
print(f'NBC News articles:           {len(nbc):,}')
print(f'  - Missing title:           {nbc["title"].isna().sum()} (pages no longer load)')
print(f'  - Missing author:          {nbc["author"].isna().sum()} (live blog pages)')
print(f'  - Missing datetime:        {nbc["datetime_posted"].isna().sum()} (live blog pages)')
print()
print(f'Combined total:              {len(fox) + len(nbc):,}')
print(f'Usable after dropping title NAs: {len(fox) + len(nbc) - nbc["title"].isna().sum():,}')
print()
print('Date range (Fox):  ', pd.to_datetime(fox['datetime_posted'], utc=True, errors='coerce').dropna().dt.date.agg(['min', 'max']).to_dict())
print('Date range (NBC):  ', pd.to_datetime(nbc['datetime_posted'], utc=True, errors='coerce').dropna().dt.date.agg(['min', 'max']).to_dict())

══ FINAL SCRAPE SUMMARY ══════════════════════════════
Fox News articles:           2,000
  - Missing subtitle:        13 (articles with no dek)
  - Missing title:           0

NBC News articles:           1,805
  - Missing title:           4 (pages no longer load)
  - Missing author:          96 (live blog pages)
  - Missing datetime:        94 (live blog pages)

Combined total:              3,805
Usable after dropping title NAs: 3,801

Date range (Fox):   {'min': datetime.date(2020, 1, 17), 'max': datetime.date(2025, 3, 16)}
Date range (NBC):   {'min': datetime.date(2020, 1, 6), 'max': datetime.date(2026, 4, 7)}
